
# 4.3. Métodos Especiales de `unittest.TestCase`

En este cuaderno aprenderemos los métodos especiales:

- `setUp()`
- `tearDown()`

Estos métodos permiten preparar y limpiar el entorno de pruebas, conocido como **fixture**.

Además del ejemplo clásico de Beatles y SQLite, aplicaremos estos conceptos a la aplicación **AppTickets** para mostrar casos reales de negocio.



## ¿Qué es un fixture?

Un fixture es el entorno necesario para ejecutar una prueba.

Por ejemplo:

- Crear objetos antes de probarlos.
- Abrir archivos.
- Crear tablas temporales.
- Abrir conexiones a bases de datos.
- Iniciar servidores.
- Preparar datos de prueba.

Los métodos especiales de `unittest.TestCase` ayudan a automatizar estas tareas.


In [ ]:

import unittest
import sqlite3
import os
from abc import ABC, abstractmethod
from functools import wraps
from datetime import datetime
import re



## Ejemplo clásico: Beatles y SQLite en memoria

SQLite permite crear bases de datos temporales en memoria usando:

```python
sqlite3.connect(':memory:')
```

La base desaparece cuando termina la ejecución.


In [ ]:

class Beatles:

    def __init__(self):
        self._connection = sqlite3.connect(':memory:')
        self._cursor = self._connection.cursor()

    def create(self):
        self._cursor.execute('''
        CREATE TABLE beatles (
            fname TEXT,
            lname TEXT,
            nickname TEXT
        )
        ''')

    def insert(self):
        miembros = [
            ('John', 'Lennon', 'The Smart One'),
            ('Paul', 'McCartney', 'The Cute One'),
            ('George', 'Harrison', 'The Funny One'),
            ('Ringo', 'Starr', 'The Quiet One')
        ]

        self._cursor.executemany(
            'INSERT INTO beatles VALUES (?,?,?)',
            miembros
        )

    def select(self, query):
        self._cursor.execute(query)
        return self._cursor.fetchall()

    def select_one(self, query):
        self._cursor.execute(query)
        return self._cursor.fetchone()

    def close(self):
        self._cursor.close()
        self._connection.close()



## Uso de `setUp()` y `tearDown()`

Observa que:

- `setUp()` se ejecuta antes de cada prueba.
- `tearDown()` se ejecuta después de cada prueba.

Para demostrarlo imprimiremos mensajes.


In [ ]:

class TestBeatles(unittest.TestCase):

    def setUp(self):
        print("Setting up")
        self.beatles = Beatles()
        self.beatles.create()
        self.beatles.insert()

    def tearDown(self):
        print("Tearing down")
        self.beatles.close()

    def test_select(self):
        resultado = self.beatles.select(
            "SELECT * FROM beatles"
        )

        self.assertIsInstance(resultado, list)

    def test_select_one(self):
        resultado = self.beatles.select_one(
            "SELECT * FROM beatles"
        )

        self.assertIsInstance(resultado, tuple)


suite = unittest.TestLoader().loadTestsFromTestCase(TestBeatles)
runner = unittest.TextTestRunner(verbosity=2)
runner.run(suite)



# Aplicación AppTickets

Ahora aplicaremos exactamente la misma idea a una aplicación orientada a objetos.


In [ ]:

def registrar_accion(funcion):

    @wraps(funcion)
    def envoltura(*args, **kwargs):

        objeto = args[0]
        resultado = funcion(*args, **kwargs)

        if hasattr(objeto, "_historial"):
            objeto._historial.append({
                "fecha_hora": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                "metodo": funcion.__name__,
                "estado": objeto.estado
            })

        return resultado

    return envoltura


class TicketBase(ABC):

    @abstractmethod
    def atender(self):
        pass


class Ticket(TicketBase):

    ESTADOS_VALIDOS = [
        "Abierto",
        "En proceso",
        "Cerrado"
    ]

    def __init__(self, folio, titulo, descripcion):

        self._historial = []

        self.folio = folio
        self.titulo = titulo
        self.descripcion = descripcion
        self.estado = "Abierto"

    @property
    def folio(self):
        return self._folio

    @folio.setter
    def folio(self, valor):

        if not self.es_folio_valido(valor):
            raise ValueError("Folio inválido")

        self._folio = valor

    @staticmethod
    def es_folio_valido(folio):
        return bool(
            re.fullmatch(r"^TK-?\d{3}$", folio)
        )

    @registrar_accion
    def atender(self):
        self.estado = "En proceso"

    @registrar_accion
    def cerrar(self):
        self.estado = "Cerrado"


class TicketSoporte(Ticket):

    @registrar_accion
    def atender(self):
        self.estado = "En proceso"
        return "Mesa de ayuda"



## Fixture aplicado a AppTickets

En lugar de crear un ticket dentro de cada prueba, podemos hacerlo una sola vez en `setUp()`.


In [ ]:

class TestTicketFixture(unittest.TestCase):

    def setUp(self):

        print("Preparando ticket")

        self.ticket = TicketSoporte(
            "TK-001",
            "Error acceso",
            "Usuario no puede entrar"
        )

    def tearDown(self):

        print("Liberando recursos")

        self.ticket = None

    def test_estado_inicial(self):

        self.assertEqual(
            self.ticket.estado,
            "Abierto"
        )

    def test_atender(self):

        self.ticket.atender()

        self.assertEqual(
            self.ticket.estado,
            "En proceso"
        )


suite = unittest.TestLoader().loadTestsFromTestCase(
    TestTicketFixture
)

runner = unittest.TextTestRunner(
    verbosity=2
)

runner.run(suite)



## Ventajas de usar `setUp()`

Sin `setUp()`:

```python
def test_1():
    ticket = TicketSoporte(...)

def test_2():
    ticket = TicketSoporte(...)
```

Con `setUp()`:

```python
def setUp():
    self.ticket = TicketSoporte(...)

def test_1():
    ...

def test_2():
    ...
```

Beneficios:

- Menos código repetido.
- Mayor claridad.
- Más fácil mantenimiento.
- Todas las pruebas parten del mismo escenario.



## Uso real de `tearDown()`

En proyectos reales suele utilizarse para:

- Cerrar archivos.
- Eliminar archivos temporales.
- Cerrar conexiones SQL.
- Cerrar conexiones de red.
- Eliminar datos de prueba.
- Detener servicios temporales.

La idea es dejar limpio el entorno después de cada prueba.


In [ ]:

class TestArchivoTemporal(unittest.TestCase):

    def setUp(self):

        self.archivo = "temporal_test.txt"

        with open(self.archivo, "w", encoding="utf-8") as f:
            f.write("Hola mundo")

    def tearDown(self):

        if os.path.exists(self.archivo):
            os.remove(self.archivo)

    def test_archivo_existe(self):

        self.assertTrue(
            os.path.exists(self.archivo)
        )


suite = unittest.TestLoader().loadTestsFromTestCase(
    TestArchivoTemporal
)

runner = unittest.TextTestRunner(
    verbosity=2
)

runner.run(suite)



# Conclusión

`setUp()` y `tearDown()` son fundamentales cuando las pruebas necesitan recursos compartidos.

Ideas clave:

1. `setUp()` se ejecuta antes de cada prueba.
2. `tearDown()` se ejecuta después de cada prueba.
3. Ambos forman parte del fixture.
4. Permiten reutilizar código.
5. Reducen duplicación.
6. Facilitan pruebas con bases de datos, archivos y objetos complejos.

Frase para clase:

> Una buena prueba no solo valida resultados; también prepara y limpia correctamente su entorno de trabajo.
